# 2026 DEFRA, local council and Urban Observatory sensor, data analysis

Following a meeting with council member, Michael Terry, we were made aware of NCC's restrictions in only using calibrated DEFRA-approved and -owned AURN instrumentation and locally-managed automatic monitoring sensors. The Urban Observatory (UO) has a collection of precison MONITOR sensors and MESH sensors. The UO MESH sensors are not calibrated and not approved by DEFRA and therefore NCC cannot use then. We need to verify that the UO MONITOR sensor data agrees with that of DEFRA and AURN sensors. We begin by comparing the data from UO MESH and MONITOR sensors against the DEFRA and NCC sensors. Any and all analysis has been conducted on 2026 data.

For the purposes of this report we will use the following acronyms:
- DEFRA-owned and approved AURN sensors (DEFRA)
- Locally managed automatic monitoring sensors (Local)
- UO precision MONITOR sensors (UO-Mon)
- UO MESH sensors (UO-Mesh)

## Summary

We have highlighted the following as areas for priory action/attention:
- For PM2.5 data, the UO-Mon and UO-Mesh sensors have low agreement. We will proceed with using the UO-Mon sensors only.
- Some UO-Mesh and UO-Mon sensors have faulty wind direction and speed data, shown as constant over the course of a day, week and month, we have taken some decommisoned sensor data from 2025 to produce analysis.
- DEFRA Newcastle Centre sensor is systematically rounding PM2.5 values to nearest integer, this will be brought to NCC's attention.
- To validate the UO-Mon sensors we compared them to their closest DEFRA/Local sensors, all UO-Mon sensors compared had very good agreement with their closest DEFRA/Local sensor. Some UO-Mon sensors were compared with further (in distance) DEFRA/Local sensors in order to compare all DEFRA/Local sensors. We found increase in distance to change the relationship between sensors to resemble an exponenital. This was not fully consistent and we would appreciate some expert UO opinion on this matter.

The report is structured as follows:
- Long time interval analysis, March 2026, comparing PM2.5 concentrations from closest precision sensors (DEFRA/Local/UO-Mon) to UO-Mesh sensors and closest DEFRA/Local sensors to UO-Mon sensors. March 2026 reasonably demonstrates a longer time period where these sensors could differ in 15min - 1hr intervalled collection. We chose month that was relatively recent with minimal missing sensors and values. Caveat: there is a short time period between 19th-21st where data is missing from all sensors.
- Short time interval analysis, 30th March - 5th April 2026, comparing PM2.5 concentrations from closest UO-Mon sensors to UO-Mesh sensors and NOx and NO2 concentrations from same sensors for DEFRA, Local and UO-Mon sensors. We chose a shorter time scale for this analysis as we wanted to discern if the UO-Mesh sensors showed any correlation and temporal similarities (althought temporal analysis has not been include in this report) to their closest UO-Mon sensors. If any of the UO-Mesh sensors had shown correlation to the UO-Mon sensors, we could still use them for a shorter time period indicator system. NOx and NO2 concentrations degrade very fast, so only a short time scale is appropriate for their analysis case.
- Extraneous factors affecting PM2.5 concentrations were also analysed over long time scale, March 2025. First we examine wind effects, comparing speed and direction to concentration in 2026 was not possible as stated in the above reasoning. With 2025 data, we take old, decommisioned sensors and perform similar analysis with closest UO-Mon sensors and Local sensors. DEFRA sensors were not considered as they are situated further away. We are continuing to compare other factors such as temperature and elevation.


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import seaborn as sns
import uo_pyfetch
import datetime
from IPython.display import display, HTML
import plotly.graph_objects as go
from geopy.distance import geodesic
from statsmodels.tsa.stattools import acf
import math
from shapely.geometry import Point, LineString
import rasterio
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def to_float64(X):
    """
    Convert input data to float64.

    GPflow requires input data to be in float64 format.
    This function ensures compatibility and prevents dtype errors.
    """
    return np.asarray(X, dtype=np.float64)

uo_name_changes = pd.read_csv("naming.csv")
uo_mapping = dict(
        zip(uo_name_changes["UO_MESH_MON"],
            uo_name_changes["new_name"])
    )

In [ ]:
def closest_sensor(locations_1, locations_2):
    
    results = []
        
    for _, row_1 in locations_1.iterrows():

        coords_1 = (
            row_1['Lat'],
            row_1['Long']
        )

        for _, row_2 in locations_2.iterrows():

            coords_2 = (
                row_2['Lat'],
                row_2['Long']
            )

            distance_euclid = math.dist(coords_1, coords_2)
            distance_km = geodesic(coords_1, coords_2).km

            results.append({
                'Sensor 1': row_1['Sensor_Name'],
                'Sensor 2': row_2['Sensor_Name'],
                'Distance km': distance_km,
                'Bird Flight Distance': distance_euclid
            })

    distance_df = pd.DataFrame(results)

    closest_sensors_df = (
        distance_df.loc[
            distance_df.groupby('Sensor 1')['Distance km'].idxmin()
        ]
    )
        
    return(closest_sensors_df)

In [ ]:
def timeframe_PM25_2026(date_start, date_end):

    PM25_2026 = pd.read_csv("2026uptoMay27-PM25-UO.csv") 
    DEFRA_2026 = pd.read_csv("2026uptoMay13-PM25-DEFRA.csv") 
    local_2026 = pd.read_csv("2026uptoMay13-PM25-local.csv")
    precision_list = [DEFRA_2026, local_2026] 

    PM25_2026["Timestamp"] = pd.to_datetime(PM25_2026["Timestamp"], errors="coerce")

    for df in precision_list: 
        date = df['Date'].astype(str) 
        time = df['Time'].astype(str) 
        mask_24 = time.str.startswith('24:') 
        # fix time first 
        time_fixed = time.str.replace(r'^24:', '00:', regex=True) 
        # combine as strings 
        combined = date + ' ' + time_fixed 
        # parse AFTER fixing 
        ts = pd.to_datetime(combined, dayfirst=True) 
        # now shift ONLY those originally with 24:00 
        ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D') 
        df['Timestamp'] = ts 
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce") 

    uo = PM25_2026[
        (PM25_2026["Timestamp"] >= date_start) &
        (PM25_2026["Timestamp"] < date_end)
    ]

    defra = DEFRA_2026[
        (DEFRA_2026["Timestamp"] >= date_start) &
        (DEFRA_2026["Timestamp"] < date_end)
    ]

    local = local_2026[
        (local_2026["Timestamp"] >= date_start) &
        (local_2026["Timestamp"] < date_end)
    ]

    defra["Sensor_Name"] = "DEFRA-" + defra["Sensor_Name"].astype(str)
    local["Sensor_Name"] = "Local-" + local["Sensor_Name"].astype(str)
    uo["Sensor_Name"] = (
        uo["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo["Sensor_Name"])
    )

    sensor_locations = (uo[['Sensor_Name', 'Sensor_Centroid_Longitude', 'Sensor_Centroid_Latitude']] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True) 
                    .rename(columns={'Sensor_Centroid_Longitude': 'Long'}) 
                    .rename(columns={'Sensor_Centroid_Latitude': 'Lat'}) 
                    ) 

    mesh_locations = (sensor_locations[sensor_locations['Sensor_Name']
                    .str.contains('Mesh', na=False)]
                    .reset_index(drop=True) 
                    .drop_duplicates(subset='Sensor_Name') 
                    ) 

    monitor_locations = (sensor_locations[sensor_locations['Sensor_Name']
                                        .str.contains('Mon', na=False)] 
                                        .reset_index(drop=True) 
                                        .drop_duplicates(subset='Sensor_Name') 
                                        ) 

    defra_locations = (defra[['Sensor_Name', 'Long', 'Lat']] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True) 
                    ) 

    local_locations = (local[['Sensor_Name', 'Long', "Lat"]] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True)
                    ) 

    precision_locations = pd.concat(
        [
            defra_locations,
            local_locations,
            monitor_locations
        ],
        ignore_index=True
    )

    all_defra_locations = pd.concat(
        [
            defra_locations,
            local_locations
        ],
        ignore_index=True
    )

    closest_precision_mesh = closest_sensor(
        precision_locations,
        mesh_locations
    )

    closest_defra_monitor = closest_sensor(
        all_defra_locations,
        monitor_locations
    )

    closest_mesh_monitor = closest_sensor(
        mesh_locations,
        monitor_locations
    )

    return {
        'uo': uo,
        'defra': defra,
        'local': local,
        'mesh_locations': mesh_locations,
        'monitor_locations': monitor_locations,
        'defra_locations': defra_locations,
        'local_locations': local_locations,
        'precision_locations': precision_locations,
        'all_defra_locations': all_defra_locations,
        'closest_precision_mesh': closest_precision_mesh,
        'closest_defra_monitor': closest_defra_monitor,
        'closest_mesh_monitor': closest_mesh_monitor
    }

In [ ]:
def timeframe_NO_2026(date_start, date_end):

    NOx_2026 = pd.read_csv("2026uptoMay27-NOx-UO.csv") 
    DEFRA_2026_NO = pd.read_csv("2026uptoMay13-NO-DEFRA.csv") 
    local_2026_NO = pd.read_csv("2026uptoMay13-NO-local.csv")
    precision_list = [DEFRA_2026_NO, local_2026_NO] 

    NOx_2026["Timestamp"] = pd.to_datetime(NOx_2026["Timestamp"], errors="coerce")

    for df in precision_list: 
        date = df['Date'].astype(str) 
        time = df['Time'].astype(str) 
        mask_24 = time.str.startswith('24:') 
        # fix time first 
        time_fixed = time.str.replace(r'^24:', '00:', regex=True) 
        # combine as strings 
        combined = date + ' ' + time_fixed 
        # parse AFTER fixing 
        ts = pd.to_datetime(combined, dayfirst=True) 
        # now shift ONLY those originally with 24:00 
        ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D') 
        df['Timestamp'] = ts 
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

    uo = NOx_2026[
        (NOx_2026["Timestamp"] >= date_start) &
        (NOx_2026["Timestamp"] < date_end)
    ]

    defra = DEFRA_2026_NO[
        (DEFRA_2026_NO["Timestamp"] >= date_start) &
        (DEFRA_2026_NO["Timestamp"] < date_end)
    ]

    local = local_2026_NO[
        (local_2026_NO["Timestamp"] >= date_start) &
        (local_2026_NO["Timestamp"] < date_end)
    ]

    defra["Sensor_Name"] = "DEFRA-" + defra["Sensor_Name"].astype(str)
    local["Sensor_Name"] = "Local-" + local["Sensor_Name"].astype(str)
    uo["Sensor_Name"] = (
        uo["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo["Sensor_Name"])
    )
    
    return {
        'uo': uo,
        'defra': defra,
        'local': local
    }

In [ ]:
def timeframe_PM25_wind_2025(date_start, date_end):

    PM25_2025 = pd.read_csv('2025-PM25-UO.csv')
    PM25_2025_local = pd.read_csv('2025March-PM25-local.csv')
    westdenton_2025_wind = pd.read_csv('2025-wind-UO_westdenton.csv')
    birtley_2025_wind = pd.read_csv('2025-wind-UO_birtley.csv')

    PM25_2025["Timestamp"] = pd.to_datetime(PM25_2025["Timestamp"], errors="coerce")
    westdenton_2025_wind["Timestamp"] = pd.to_datetime(westdenton_2025_wind["Timestamp"], errors="coerce")
    birtley_2025_wind["Timestamp"] = pd.to_datetime(birtley_2025_wind["Timestamp"], errors="coerce")
    
    date = PM25_2025_local['Date'].astype(str) 
    time = PM25_2025_local['Time'].astype(str) 
    mask_24 = time.str.startswith('24:') 
    # fix time first 
    time_fixed = time.str.replace(r'^24:', '00:', regex=True) 
    # combine as strings 
    combined = date + ' ' + time_fixed 
    # parse AFTER fixing 
    ts = pd.to_datetime(combined, dayfirst=True) 
    # now shift ONLY those originally with 24:00 
    ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D') 
    PM25_2025_local['Timestamp'] = ts 
    PM25_2025_local["Timestamp"] = pd.to_datetime(PM25_2025_local["Timestamp"], errors="coerce")

    uo = PM25_2025[
        (PM25_2025["Timestamp"] >= date_start) &
        (PM25_2025["Timestamp"] < date_end)
    ]

    local = PM25_2025_local[
        (PM25_2025_local["Timestamp"] >= date_start) &
        (PM25_2025_local["Timestamp"] < date_end)
    ]

    wd = westdenton_2025_wind[
        (westdenton_2025_wind["Timestamp"] >= date_start) &
        (westdenton_2025_wind["Timestamp"] < date_end)
    ]

    birt = birtley_2025_wind[
        (birtley_2025_wind["Timestamp"] >= date_start) &
        (birtley_2025_wind["Timestamp"] < date_end)
    ]

    wind = pd.concat(
        [wd, birt]
    )
    
    local["Sensor_Name"] = "Local-" + local["Sensor_Name"].astype(str)
    uo["Sensor_Name"] = (
        uo["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo["Sensor_Name"])
    )
    rename_wind_sensors = {
    "PER_EMLFLOOD_UO-BIRTLEYFS": "UO-WIND-birtley",
    "PER_EMLFLOOD_UO-WDENTONFS": "UO-WIND-wdenton"
    }
    wind["Sensor_Name"] = wind["Sensor_Name"].replace(rename_wind_sensors)

    wind_locations = (wind[['Sensor_Name', 'Sensor_Centroid_Longitude', 'Sensor_Centroid_Latitude']]
                .dropna()
                .drop_duplicates(subset='Sensor_Name')
                .rename(columns={'Sensor_Centroid_Longitude': 'Long',
                                'Sensor_Centroid_Latitude': 'Lat'
                                })
                )
    
    monitor_locations = (uo[uo['Sensor_Name']
                                        .str.contains('Mon', na=False)] 
                                        .reset_index(drop=True) 
                                        .drop_duplicates(subset='Sensor_Name')
                                        .rename(columns={'Sensor_Centroid_Longitude': 'Long',
                                                         'Sensor_Centroid_Latitude': 'Lat'})
                                        ) 
    
    local_locations = (local[['Sensor_Name', 'Long', "Lat"]] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True)
                    )

    closest_wind_monitor = closest_sensor(
        wind_locations,
        monitor_locations
    )

    closest_wind_local = closest_sensor(
        wind_locations,
        local_locations
    )

    return {
        'uo': uo,
        'wind': wind,
        'local': local,
        'wind_locations': wind_locations,
        'monitor_locations': monitor_locations,
        'local_locations': local_locations,
        'closest_wind_monitor': closest_wind_monitor,
        'closest_wind_local': closest_wind_local
    }

In [ ]:
march_2026 = timeframe_PM25_2026("2026-03-01", "2026-03-31")

defra_names = set(march_2026['defra_locations']['Sensor_Name'].unique())
local_names = set(march_2026['local_locations']['Sensor_Name'].unique())
monitor_names = set(march_2026['monitor_locations']['Sensor_Name'].unique()) 
mesh_names = set(march_2026['mesh_locations']['Sensor_Name'].unique()) 

In [ ]:
march_week_2026 = timeframe_PM25_2026("2026-03-30","2026-04-05")

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

## 1. Maps

The three maps in Figure 1 show closest precision sensors (DEFRA/Local/UO-Mon) to UO-Mesh sensors, closest DEFRA/Local sensors to UO-Mon sensors, and closest UO-Mon sensors to UO-Mesh sensors. These pairs will be used to analyse correlations between closest sensors, validate UO-Mon sensors and understand if UO-Mesh sensors can be used in practice. The distances (km and birds flight) between these pairs are stated in Section 2 correlational analysis.

In [ ]:
def plot_sensor_pairs_map(
    pair_df,
    locations_1,
    locations_2,
    all_locations=None,
    label_1='Sensor 1',
    label_2='Sensor 2',
    title='Sensor Pair Map',
    figsize=(15, 15),
    ax=None
):

    merged = (
        pair_df
        .merge(
            locations_1[
                ['Sensor_Name', 'Long', 'Lat']
            ],
            left_on='Sensor 1',
            right_on='Sensor_Name',
            how='left'
        )
        .rename(columns={
            'Long': 'Long_1',
            'Lat': 'Lat_1'
        })
        .drop(columns='Sensor_Name')
    )

    merged = (
        merged
        .merge(
            locations_2[
                ['Sensor_Name', 'Long', 'Lat']
            ],
            left_on='Sensor 2',
            right_on='Sensor_Name',
            how='left'
        )
        .rename(columns={
            'Long': 'Long_2',
            'Lat': 'Lat_2'
        })
        .drop(columns='Sensor_Name')
    )

    line_geoms = []

    for _, row in merged.iterrows():

        if (
            pd.isna(row['Long_1'])
            or
            pd.isna(row['Long_2'])
        ):
            continue

        line_geoms.append(
            LineString([
                (
                    row['Long_1'],
                    row['Lat_1']
                ),
                (
                    row['Long_2'],
                    row['Lat_2']
                )
            ])
        )

    lines_gdf = gpd.GeoDataFrame(
        geometry=line_geoms,
        crs='EPSG:4326'
    )

    paired_1 = gpd.GeoDataFrame(
        merged,
        geometry=gpd.points_from_xy(
            merged['Long_1'],
            merged['Lat_1']
        ),
        crs='EPSG:4326'
    )

    paired_2 = gpd.GeoDataFrame(
        merged,
        geometry=gpd.points_from_xy(
            merged['Long_2'],
            merged['Lat_2']
        ),
        crs='EPSG:4326'
    )

    background_gdf = None

    if all_locations is not None:

        paired_names = set(
            pair_df['Sensor 1']
        ).union(
            set(pair_df['Sensor 2'])
        )

        background = (
            all_locations[
                ~all_locations[
                    'Sensor_Name'
                ].isin(paired_names)
            ]
        )

        background_gdf = gpd.GeoDataFrame(
            background,
            geometry=gpd.points_from_xy(
                background['Long'],
                background['Lat']
            ),
            crs='EPSG:4326'
        )

    paired_1 = paired_1.to_crs(3857)
    paired_2 = paired_2.to_crs(3857)
    lines_gdf = lines_gdf.to_crs(3857)

    if background_gdf is not None:
        background_gdf = (
            background_gdf
            .to_crs(3857)
        )
        
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)

    # other sensors
    if background_gdf is not None:

        background_gdf.plot(
            ax=ax,
            markersize=20,
            alpha=0.35,
            label='Other sensors'
        )

    # connecting lines
    lines_gdf.plot(
        ax=ax,
        linewidth=1.5,
        alpha=0.7,
        label='Pair links'
    )

    # paired sensors
    paired_1.plot(
        ax=ax,
        markersize=80,
        label=label_1
    )

    paired_2.plot(
        ax=ax,
        markersize=80,
        marker='^',
        label=label_2
    )

    # basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron
    )

    ax.set_title(title, fontsize=16)
    ax.legend(prop={'size': 14})
    ax.set_axis_off()

Figure 1: Maps of (left to right) closest precision sensors (DEFRA/Local/UO-Mon) to UO-Mesh sensors, closest DEFRA/Local sensors to UO-Mon sensors, and closest UO-Mon sensors to UO-Mesh sensors.

In [ ]:
fig = plt.figure(figsize=(20, 14))

gs = fig.add_gridspec(2, 2)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, :])
]

all_sensors = pd.concat(
    [
        march_2026['precision_locations'],
        march_2026['mesh_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    march_2026['closest_precision_mesh'],
    march_2026['precision_locations'],
    march_2026['mesh_locations'],
    all_locations=all_sensors,
    label_1='Precision (DEFRA/Local/UO-Mon)',
    label_2='UO-Mesh',
    title='Closest Precision to UO-Mesh Sensors',
    ax=axes[0]
)

all_sensors = pd.concat(
    [
        march_2026['all_defra_locations'],
        march_2026['monitor_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    march_2026['closest_defra_monitor'],
    march_2026['all_defra_locations'],
    march_2026['monitor_locations'],
    all_locations=all_sensors,
    label_1='DEFRA/Local',
    label_2='UO-Mon',
    title='Closest DEFRA/Local to UO-Mon Sensors',
    ax=axes[1]
)

all_sensors = pd.concat(
    [
        march_2026['monitor_locations'],
        march_2026['mesh_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    march_week_2026['closest_mesh_monitor'],
    march_week_2026['mesh_locations'],
    march_week_2026['monitor_locations'],
    all_locations=all_sensors,
    label_1='UO-Mesh',
    label_2='UO-Mon',
    title='Closest UO-Mon to UO-Mesh Sensors',
    ax=axes[2]
)

plt.tight_layout()
plt.show()

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

## 2. Monthly data

We took March 2026 to look at correlation comparisons between the closest precision (DEFRA/Local/UO-Mon) to UO-Mesh sensors and closest DEFRA/Local sensors to UO-Mon sensors. March was chosen as the dataset had reliable consistent data, i.e. minimal missing values and missing sensor data.

### 2.1 PM2.5 comparisons between close sensors

Closest precision (DEFRA/Local/UO-Mon) to UO-Mesh sensors, Figure 2 below shows scatter plots of these relationships. These show little to no reliability of the UO mesh sensors.

In [ ]:
valid_rows = []

for _, row in march_2026['closest_precision_mesh'].iterrows():

    precision_name = row['Sensor 1']
    mesh_name = row['Sensor 2']

    if precision_name in defra_names:
        precision_df = march_2026['defra']
    elif precision_name in local_names:
        precision_df = march_2026['local']
    elif precision_name in monitor_names:
        precision_df = (march_2026['uo'][march_2026['uo']['Sensor_Name'].str.contains('Mon', na=False)]
            .reset_index(drop=True)
            .rename(columns={'Value': 'PM2.5'
                             })
                             ) 

    # subset
    precision = (
        precision_df[
            precision_df['Sensor_Name'] == precision_name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
        .rename(columns={'PM2.5': 'precision_pm25'})
    )

    precision['precision_pm25'] = pd.to_numeric(
        precision['precision_pm25'],
        errors='coerce'
    )

    # matching mesh subset
    mesh = (
        march_2026['uo'][
            march_2026['uo']['Sensor_Name'] == mesh_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'mesh_pm25'})
    )

    mesh['mesh_pm25'] = pd.to_numeric(
        mesh['mesh_pm25'], 
        errors='coerce')

    # nearest timestamp match
    merged = pd.merge_asof(
        precision,
        mesh,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('10min')
    )

    merged = merged.dropna(subset=['precision_pm25', 'mesh_pm25'])

    if not merged.empty:
        valid_rows.append((row, precision_name, mesh_name, merged))

Figure 2: PM2.5 concentrations comparisons between UO-Mesh and closest DEFRA/Local/UO-Mon sensors over long time interval.

In [ ]:
n_plots = len(march_2026['closest_precision_mesh'])
ncols = 4
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, 12),
    constrained_layout=True
)

axes = axes.flatten()
scatter_ref = None

for i, (row, precision_name, mesh_name, merged) in enumerate(valid_rows):

    ax = axes[i]

    corr = merged[['precision_pm25', 'mesh_pm25']].corr().iloc[0, 1]

    scatter_ref = ax.scatter(
        merged['precision_pm25'],
        merged['mesh_pm25'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged[['precision_pm25', 'mesh_pm25']].min().min(),
    merged[['precision_pm25', 'mesh_pm25']].max().max()
    ]

    ax.plot(lims, lims)

    ax.set_xlabel('Precision PM2.5')
    ax.set_ylabel('UO-Mesh PM2.5')
    ax.set_title(f'{precision_name}\nvs\n{mesh_name}')
    ax.grid(True)

# hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.show()

Figure 3 shows closest UO-Mon sensors to DEFRA/Local sensors, scatter plots and time series, where only the closest UO-Mon sensor is included for each DEFRA/Local sensor. This hopes to validate the UO-Mon sensor data and from both sets of plots we see very good agreement between the sensors. Figure 4 shows the same comparison as Figure 3 however the UO-Mon sensors under consideration (UO-Mon Gateshead Tyne Bridge and UO-Mon RVI) have already been compared to other DEFRA/Local sensors in Figure 3 that are much closer, we see worse agreement for two of the sensor comparisons here which is expected due to the distance. A summary table of correlational information and distances between sensors is given in Table 1 from which we see high correlation between all close sensors. We use both Pearson's and Spearman's correlation coefficents. Pearson's evaluates linear relationships and proportional change between variables while Spearman's monotonic relationships which dicerns if variables are changing together, not necessarily at a constant rate as Pearson's would let us assume. It is important to remember that other relationship (non-linear) might also exist as we see in the scaling differences between Loal A1 Dunston sensor and Local Gateshead Lychgate Court sensors with their closest UO-Mon sensor. This could be attributed to their further distance from thier closest UO-Mon if it were not for DEFRA Newcastle Cradlewell Roadside case which is further away from it's closest UO-Mon than the Local Gateshead Lychgate Court sensor. It could also be attributed to the fact that we are comparing the same monitor sensor for 4 of the DEFRA/Local sensors as there are no closer UO-Mon sensors in the area. Another issue is the case of DEFRA Newcastle Centre sensor, this is systematically rounding values to the closest integer, we will see this more clearly when analysing weekly data but here you can spot it in the time series plot.

In [ ]:
closest_pairs = (
    march_2026['closest_defra_monitor']
    .sort_values('Distance km')
    .drop_duplicates(subset='Sensor 2', keep='first')
)

repeat_pairs = (
    march_2026['closest_defra_monitor']
    .loc[
        ~march_2026['closest_defra_monitor'].index.isin(closest_pairs.index)
    ]
)

In [ ]:
summary_results = []

def make_valid_rows(pair_df):

    valid_rows = []

    for _, row in pair_df.iterrows():

        precision_name = row['Sensor 1']
        monitor_name = row['Sensor 2']

        if precision_name in defra_names:
            precision_df = march_2026['defra']
        elif precision_name in local_names:
            precision_df = march_2026['local']

        precision = (
            precision_df[
                precision_df['Sensor_Name'] == precision_name
            ][['Timestamp', 'PM2.5']]
            .sort_values('Timestamp')
            .rename(columns={'PM2.5': 'precision_pm25'})
        )

        precision['precision_pm25'] = pd.to_numeric(
            precision['precision_pm25'],
            errors='coerce'
        )

        monitor = (
            march_2026['uo'][
                march_2026['uo']['Sensor_Name'] == monitor_name
            ][['Timestamp', 'Value']]
            .sort_values('Timestamp')
            .rename(columns={'Value': 'monitor_pm25'})
        )

        monitor['monitor_pm25'] = pd.to_numeric(
            monitor['monitor_pm25'],
            errors='coerce'
        )

        merged = pd.merge_asof(
            precision,
            monitor,
            on='Timestamp',
            direction='nearest',
            tolerance=pd.Timedelta('1min')
        ).dropna(subset=['precision_pm25', 'monitor_pm25'])

        if not merged.empty:
            valid_rows.append((row, precision_name, monitor_name, merged))

    return valid_rows

closest_valid_rows = make_valid_rows(closest_pairs)
repeat_valid_rows = make_valid_rows(repeat_pairs)

Figure 3: Comparison of PM2.5 concentrations measured by each DEFRA/Local sensor and its closest UO-Mon sensor over the study period. Only the closest UO-Mon sensor is included for each DEFRA/Local sensor.

In [ ]:
n_pairs = len(closest_valid_rows)

ncols_pairs = 2
nrows_pairs = math.ceil(n_pairs / ncols_pairs)

fig, axes = plt.subplots(
    nrows=nrows_pairs,
    ncols=ncols_pairs * 2,   # 2 plots per pair
    figsize=(20, 4.5 * nrows_pairs),
    constrained_layout=True
)

for i in range(n_pairs, nrows_pairs * ncols_pairs):
        pair_row = i // ncols_pairs
        pair_col = i % ncols_pairs

        ax1 = axes[pair_row, pair_col * 2]
        ax2 = axes[pair_row, pair_col * 2 + 1]

        ax1.set_visible(False)
        ax2.set_visible(False)

axes = np.atleast_2d(axes)

scatter_ref = None

for i, (row, precision_name, monitor_name, merged) in enumerate(closest_valid_rows):

    pair_row = i // ncols_pairs
    pair_col = i % ncols_pairs

    ax1 = axes[pair_row, pair_col * 2]
    ax2 = axes[pair_row, pair_col * 2 + 1]

    # Pearson correlation
    pearson_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'DEFRA/Local': precision_name,
    'UO-Mon': monitor_name,
    'Distance km': row['Distance km'],
    'Bird Flight Distance' : row['Bird Flight Distance'],
    'Pearsons corr': pearson_corr,
    'Spearmans corr': spearman_corr,
    })  

    scatter_ref = ax1.scatter(
        merged['precision_pm25'],
        merged['monitor_pm25'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged[['precision_pm25', 'monitor_pm25']].min().min(),
    merged[['precision_pm25', 'monitor_pm25']].max().max()
    ]

    ax1.plot(lims, lims)

    ax1.set_xlabel('DEFRA/Local PM2.5')
    ax1.set_ylabel('UO-Mon PM2.5')
    ax1.grid(True)

    ax2.plot(
        merged['Timestamp'],
        merged['precision_pm25'],
        label='DEFRA'
    )

    ax2.plot(
        merged['Timestamp'],
        merged['monitor_pm25'],
        label='UO-Mon'
    )

    ax2.set_xlabel('Time')
    ax2.set_ylabel('PM2.5')
    ax2.legend()
    ax2.grid(True)
    ax2.tick_params(axis='x', labelrotation=45)

    ax1.set_title(
    f'{precision_name} vs {monitor_name}', pad=10, x=0.99
    )

#plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 4: Comparison of PM2.5 concentrations measured by DEFRA/Local sensors and additional nearby UO-Mon sensors over the study period. These comparisons exclude the nearest UO-Mon sensor shown in Figure 3 and illustrate relationships with other matched UO-Mon sensors.

In [ ]:
n_pairs = len(repeat_valid_rows)

ncols_pairs = 2
nrows_pairs = math.ceil(n_pairs / ncols_pairs)

fig, axes = plt.subplots(
    nrows=nrows_pairs,
    ncols=ncols_pairs * 2,   # 2 plots per pair
    figsize=(20, 4.5 * nrows_pairs),
    constrained_layout=True
)

for i in range(n_pairs, nrows_pairs * ncols_pairs):
        pair_row = i // ncols_pairs
        pair_col = i % ncols_pairs

        ax1 = axes[pair_row, pair_col * 2]
        ax2 = axes[pair_row, pair_col * 2 + 1]

        ax1.set_visible(False)
        ax2.set_visible(False)

axes = np.atleast_2d(axes)

scatter_ref = None

for i, (row, precision_name, monitor_name, merged) in enumerate(repeat_valid_rows):

    pair_row = i // ncols_pairs
    pair_col = i % ncols_pairs

    ax1 = axes[pair_row, pair_col * 2]
    ax2 = axes[pair_row, pair_col * 2 + 1]

    # Pearson correlation
    pearson_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'DEFRA/Local': precision_name,
    'UO-Mon': monitor_name,
    'Distance km': row['Distance km'],
    'Bird Flight Distance' : row['Bird Flight Distance'],
    'Pearsons corr': pearson_corr,
    'Spearmans corr': spearman_corr,
    })  

    scatter_ref = ax1.scatter(
        merged['precision_pm25'],
        merged['monitor_pm25'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged[['precision_pm25', 'monitor_pm25']].min().min(),
    merged[['precision_pm25', 'monitor_pm25']].max().max()
    ]

    ax1.plot(lims, lims)

    ax1.set_xlabel('DEFRA/Local PM2.5')
    ax1.set_ylabel('UO-Mon PM2.5')
    ax1.grid(True)

    ax2.plot(
        merged['Timestamp'],
        merged['precision_pm25'],
        label='DEFRA'
    )

    ax2.plot(
        merged['Timestamp'],
        merged['monitor_pm25'],
        label='UO-Mon'
    )

    ax2.set_xlabel('Time')
    ax2.set_ylabel('PM2.5')
    ax2.legend()
    ax2.grid(True)
    ax2.tick_params(axis='x', labelrotation=45)

    ax1.set_title(
    f'{precision_name} vs {monitor_name}', pad=10, x=0.99
    )

#plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()

Just to check on this nonlinear relationship, we checked the agreement between Local-Gateshead A1 Dunston and Local-Gateshead Tyne Bridge since Figure 4 shows it with UO-Mon Geateshead Tyne Bridge (in the exact same place) so we understand where the disagreement is coming from.

In [ ]:
A1 = (
    march_2026['local'][
        march_2026['local']['Sensor_Name'] == 'Local-Gateshead A1 Dunston'
    ][['Timestamp', 'PM2.5']]
    .sort_values('Timestamp')
    .rename(columns={'PM2.5': 'A1_pm25'})
)

A1['A1_pm25'] = pd.to_numeric(
    A1['A1_pm25'],
    errors='coerce'
)

TyneB = (
    march_2026['local'][
        march_2026['local']['Sensor_Name'] == 'Local-Gateshead Tyne Bridge'
    ][['Timestamp', 'PM2.5']]
    .sort_values('Timestamp')
    .rename(columns={'PM2.5': 'TyneB_pm25'})
)

TyneB['TyneB_pm25'] = pd.to_numeric(
    TyneB['TyneB_pm25'],
    errors='coerce'
)

merged = pd.merge_asof(
    A1,
    TyneB,
    on='Timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('1min')
).dropna(subset=['A1_pm25', 'TyneB_pm25'])

fig, ax = plt.subplots(1,2,figsize=(9,4))
ax1 = ax[0]
ax2 = ax[1]

ax1.scatter(merged['A1_pm25'],
           merged['TyneB_pm25'],
           alpha=0.5,
            c='red',
            s = 20)

ax1.set_xlabel('Local-A1 Dunston PM2.5')
ax1.set_ylabel('Local-Tyne Bridge PM2.5')
ax1.set_ylim(0.7,1)
ax1.set_title('PM2.5')
ax1.grid(True)
ax1.legend()

ax2.plot(
    merged['Timestamp'],
    merged['A1_pm25'],
    label='Gateshead A1 Dunston'
)

ax2.plot(
    merged['Timestamp'],
    merged['TyneB_pm25'],
    label='Gateshead Tyne Bridge'
)

ax2.set_xlabel('Time')
ax2.set_ylabel('PM2.5')
ax2.legend()
ax2.grid(True)
ax2.tick_params(axis='x', labelrotation=45)

ax1.set_title(
f'Local Gateshead A1 Dunston vs Tyne Bridge', pad=10, x=0.99
)

plt.show()

Table 1: Distances between sensors in km and , Pearson's and Spearman's correlations for UO-Mon and closest DEFRA/Local sensors PM2.5 concentrations over long time interval.

In [ ]:
summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'DEFRA/Local',
    'UO-Mon',
    'Distance km',
    'Bird Flight Distance',
    'Pearsons corr',
    'Spearmans corr'
    
]]

summary_df = summary_df.sort_values(
    by='Pearsons corr',
    ascending=False
)

summary_df[['Distance km', 'Bird Flight Distance', 'Pearsons corr', 'Spearmans corr']] = (
    summary_df[['Distance km', 'Bird Flight Distance', 'Pearsons corr', 'Spearmans corr']].round(3)
)

summary_df.set_index('DEFRA/Local', inplace=True)

summary_df

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 5: Correlation coefficents of each sensor pair from Figure 3 and Table 1 with their bird's flight distance.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5,3.5))

ax.scatter(summary_df['Bird Flight Distance'],
           summary_df['Pearsons corr'],
           label='Pearson',
           alpha=0.7)

ax.scatter(summary_df['Bird Flight Distance'],
           summary_df['Spearmans corr'],
           label='Spearman',
           alpha=0.7)

ax.set_xlabel('Bird Flight Distance')
ax.set_ylabel('Correlation')
ax.set_ylim(0.7,1)
ax.set_title('Closest UO-Mon to DEFRA/Local sensors')
ax.grid(True)
ax.legend()

plt.show()

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

## 3. Weekly data

We took the last week (30th March-5th April) of March 2026 to look at correlation comparisons in PM2.5 concentrations between the UO-Mesh sensors to UO-Mon sensors. We also looked at correlational comparisons of PM2.5 and NOx/NO2 concentrations in the same sensors for all DEFRA, Local and UO-Mon sensors.

### 3.1 PM2.5 comparisons between close sensors

Figure 6 shows closest UO-Mon sensors to UO-Mesh sensors scatter plots. Unfortunately, only two of the UO-MonInterestingly, there are some sensors that have decent agreement with the two close UO-Mon sensors, specifically UO-Mesh West Denton and Coast Road (next to the UO-Mon Coast Road sensor). Most others show little agreement however moderate positive correlation, UO-Mesh Heaton Road and Grainger Market have particularily high correlations for both Pearson's and Spearman's (seen in Table 2). Oddly, the UO-Mesh West Denton sensor is much further away from the UO-Mon RVI sensor which is simiarly compared to the UO-Mesh Grainger Market. Some sensors show little to no correlation however UO-Mesh Wingrove with UO-Mon RVI has very low Pearson's correlation and moderate Spearman's correlation, indicating some monotonic relation between then PM2.5 concentrations. There are some intriguing and hopeful structures in the data, these vastly differ week to week although the same correlation pattern between sensors is present which will need some further analysis.

In [ ]:
defra_names = set(march_week_2026['defra_locations']['Sensor_Name'].unique())
local_names = set(march_week_2026['local_locations']['Sensor_Name'].unique())
monitor_names = set(march_week_2026['monitor_locations']['Sensor_Name'].unique()) 
mesh_names = set(march_week_2026['mesh_locations']['Sensor_Name'].unique()) 

In [ ]:
summary_results = []
valid_rows = []

for _, row in march_week_2026['closest_mesh_monitor'].iterrows():

    mesh_name = row['Sensor 1']
    monitor_name = row['Sensor 2']
    
    mesh = (
        march_week_2026['uo'][
            march_week_2026['uo']['Sensor_Name'] == mesh_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'mesh_pm25'})
    )

    mesh['mesh_pm25'] = pd.to_numeric(
        mesh['mesh_pm25'],
        errors='coerce'
    )

    monitor = (
        march_week_2026['uo'][
            march_week_2026['uo']['Sensor_Name'] == monitor_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'monitor_pm25'})
    )

    monitor['monitor_pm25'] = pd.to_numeric(
        monitor['monitor_pm25'], 
        errors='coerce')

    # nearest timestamp match
    merged = pd.merge_asof(
        mesh,
        monitor,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    ).dropna(subset=['mesh_pm25', 'monitor_pm25'])

    if not merged.empty:
        valid_rows.append((row, mesh_name, monitor_name, merged))


In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 6: PM2.5 concentrations comparisons between UO-Mesh and closest UO-Mon sensors over short time interval.

In [ ]:
n_plots = len(valid_rows)
ncols = 4
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(16, 14),
    constrained_layout=True
)

axes = axes.flatten()
scatter_ref = None

for i, (row, mesh_name, monitor_name, merged) in enumerate(valid_rows):

    ax = axes[i]

    # Pearson correlation
    pearson_corr = merged[['mesh_pm25', 'monitor_pm25']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr = merged[['mesh_pm25', 'monitor_pm25']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'UO-Mesh': mesh_name,
    'UO-Mon': monitor_name,
    'Distance km': row['Distance km'],
    'Bird Flight Distance' : row['Bird Flight Distance'],
    'Pearsons corr': pearson_corr,
    'Spearmans corr': spearman_corr,
    }) 

    scatter_ref = ax.scatter(
        merged['monitor_pm25'],
        merged['mesh_pm25'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged[['monitor_pm25', 'mesh_pm25']].min().min(),
    merged[['monitor_pm25', 'mesh_pm25']].max().max()
    ]

    ax.plot(lims, lims)

    ax.set_xlabel('UO-Mon PM2.5')
    ax.set_ylabel('UO-Mesh PM2.5')
    ax.set_title(f'{monitor_name}\nvs\n{mesh_name}')
    ax.grid(True)

# hide unused axes
for j in range(len(valid_rows), len(axes)):
    axes[j].set_visible(False)

plt.show()

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Table 2: Distances between sensors in km and bird flight, Pearson's and Spearman's correlations for UO-Mesh and closest UO-Mon sensors PM2.5 concentrations over short time interval.

In [ ]:
summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'UO-Mesh',
    'UO-Mon',
    'Distance km',
    'Bird Flight Distance',
    'Pearsons corr',
    'Spearmans corr'
]]

summary_df = summary_df.sort_values(
    by='Pearsons corr',
    ascending=False
)

summary_df[['Distance km', 'Bird Flight Distance', 'Pearsons corr', 'Spearmans corr']] = (
    summary_df[['Distance km', 'Bird Flight Distance', 'Pearsons corr', 'Spearmans corr']].round(3)
)

summary_df.set_index('UO-Mesh', inplace=True)

summary_df

### 3.2 NOx and NO2 comparisons between same sensors

First NOx and NO2 concentration comparison in the DEFRA sensors to dicern if PM2.5 has any traffic dependency in the same way that NO2 and NOx have. In Figure 7 scatter plots, we see very clearly here the systematic categorisation of the DEFRA Newcastle Centre sensor. Low correlation values can be seen for all in Table 3.

In [ ]:
march_week_2026_NO = timeframe_NO_2026("2026-03-30","2026-04-05")

defra_names = set(march_week_2026_NO['defra']['Sensor_Name'].unique())
local_names = set(march_week_2026_NO['local']['Sensor_Name'].unique())
monitor_names = (march_week_2026_NO['uo'].loc[march_week_2026_NO['uo']['Sensor_Name']
        .str.contains('Mon', na=False),
        'Sensor_Name']
    .unique()
    .tolist()
)

mesh_names = (march_week_2026_NO['uo'].loc[march_week_2026_NO['uo']['Sensor_Name']
        .str.contains('Mesh', na=False),
        'Sensor_Name']
    .unique()
    .tolist()
)

In [ ]:
summary_results = []
valid_names = []

for name in defra_names:

    pm25 = (
        march_week_2026['defra'][
            march_week_2026['defra']['Sensor_Name'] == name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    nox = (
        march_week_2026_NO['defra'][
            march_week_2026_NO['defra']['Sensor_Name'] == name
        ][['Timestamp', 'NOx', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    nox['NOx'] = pd.to_numeric(
        nox['NOx'],
        errors='coerce'
    )

    no2 = (
        march_week_2026_NO['defra'][
            march_week_2026_NO['defra']['Sensor_Name'] == name
        ][['Timestamp', 'NO2', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    no2['NO2'] = pd.to_numeric(
        no2['NO2'],
        errors='coerce'
    )

    # nearest timestamp match
    merged_nox = pd.merge_asof(
        pm25,
        nox,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_nox = merged_nox.dropna(subset=['PM2.5', 'NOx'])

    merged_no2 = pd.merge_asof(
        pm25,
        no2,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_no2 = merged_no2.dropna(subset=['PM2.5', 'NO2'])

    if not merged_nox.dropna().empty and not merged_no2.dropna().empty:
        valid_names.append((name, merged_nox, merged_no2))

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 7: Comparsion of PM2.5 and NOx and NO2 concentrations for DEFRA sensors over short time interval.

In [ ]:
n_pairs = len(valid_names)

fig, axes = plt.subplots(
    nrows=n_pairs,
    ncols=2,
    figsize=(12, 4 * n_pairs),
    constrained_layout=True
)

# if only one row
if n_pairs == 1:
    axes = np.array([axes])

scatter_ref = None

for i, (name, merged_nox, merged_no2) in enumerate(valid_names):

    ax1, ax2 = axes[i]

        # Pearson correlation
    pearson_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='pearson').iloc[0, 1]
    pearson_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='spearman').iloc[0, 1]
    spearman_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'DEFRA': name,
    'Pearsons corr NOx': pearson_corr_nox,
    'Spearmans corr NOx': spearman_corr_nox,
    'Pearsons corr NO2': pearson_corr_no2,
    'Spearmans corr NO2': spearman_corr_no2
    })

    nox_plt = ax1.scatter(
    merged_nox['PM2.5'],
    merged_nox['NOx'],
    alpha=0.5,
    c='red',
    s=20
    )

    scatter_ref = nox_plt
    # 1:1 line
    lims = [
    merged_nox[['PM2.5', 'NOx']].min().min(),
    merged_nox[['PM2.5', 'NOx']].max().max()
]

    ax1.plot(lims, lims)

    ax1.set_xlabel('PM2.5')
    ax1.set_ylabel('NOx')
    ax1.set_title(f'{name}')
    ax1.grid(True)

    no2_plt = ax2.scatter(
        merged_no2['PM2.5'],
        merged_no2['NO2'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged_no2[['PM2.5', 'NO2']].min().min(),
    merged_no2[['PM2.5', 'NO2']].max().max()
]

    ax2.plot(lims, lims)

    ax2.set_xlabel('PM2.5')
    ax2.set_ylabel('NO2')
    ax2.set_title(f'{name}')
    ax2.grid(True)

#plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()

Table 3: Pearson's and Spearman's correlations for DEFRA sensor PM2.5 concentrations and NOx, NO2 concentrations over short time interval.

In [ ]:
summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'DEFRA',
    'Pearsons corr NOx',
    'Spearmans corr NOx',
    'Pearsons corr NO2',
    'Spearmans corr NO2'
]]

summary_df = summary_df.sort_values(
    by='Pearsons corr NOx',
    ascending=False
)

summary_df[['Pearsons corr NOx', 'Spearmans corr NOx', 'Pearsons corr NO2', 'Spearmans corr NO2']] = (
    summary_df[['Pearsons corr NOx', 'Spearmans corr NOx', 'Pearsons corr NO2', 'Spearmans corr NO2']].round(3)
)

summary_df.set_index('DEFRA', inplace=True)

summary_df

In Figure 8, we can see scatter plots for NO2 comparison with PM2.5 concentration for the Local sensors as they not not collect NOx data. Moderate correlation values can be seen in Table 4 which suggest there may be some traffic dependence or possibly other factors like temperature.

In [ ]:
summary_results = []
valid_names = []

for name in local_names:

    pm25 = (
        march_week_2026['local'][
            march_week_2026['local']['Sensor_Name'] == name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    no2 = (
        march_week_2026_NO['local'][
            march_week_2026_NO['local']['Sensor_Name'] == name
        ][['Timestamp', 'NO2', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    no2['NO2'] = pd.to_numeric(
        no2['NO2'],
        errors='coerce'
    )

    merged_no2 = pd.merge_asof(
        pm25,
        no2,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_no2 = merged_no2.dropna(subset=['PM2.5', 'NO2'])

    if not merged_no2.dropna().empty:
        valid_names.append((name, merged_no2))

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 8: Comparisons of Local sensor PM2.5 and NO2 concentrations over short time interval.

In [ ]:
n_plots = len(valid_names)
ncols = 2
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12, 8),
    constrained_layout=True
)

axes = axes.flatten()
scatter_ref = no2_plt

for i, (name, merged_no2) in enumerate(valid_names):

    ax = axes[i]

    # Pearson correlation
    pearson_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'Local': name,
    'Pearsons corr NO2': pearson_corr_no2,
    'Spearmans corr NO2': spearman_corr_no2
    })

    scatter_ref = ax.scatter(
        merged_no2['PM2.5'],
        merged_no2['NO2'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged_no2[['PM2.5', 'NO2']].min().min(),
    merged_no2[['PM2.5', 'NO2']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NO2')
    ax.set_title(f'{name}')
    ax.grid(True)

# hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.show()

Table 4: Pearson's and Spearman's correlations for Local sensor PM2.5 concentrations and NO2 concentrations over short time interval.

In [ ]:
summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'Local',
    'Pearsons corr NO2',
    'Spearmans corr NO2'
]]

summary_df = summary_df.sort_values(
    by='Pearsons corr NO2',
    ascending=False
)

summary_df[['Pearsons corr NO2', 'Spearmans corr NO2']] = (
    summary_df[['Pearsons corr NO2', 'Spearmans corr NO2']].round(3)
)

summary_df.set_index('Local', inplace=True)

summary_df

In Figure 9, scatter plots for NOx comparison with PM2.5 concentration for the UO-Mon sensors for now. NO2 concentration comparisons are still in process with some data download difficulties with UO-API. In Table 5, we see that some of the UO-Mon sensors show moderate positive correlation between PM2.5 and NOx, however the other half show little to no correlation.

In [ ]:
summary_results = []
valid_names = []

for name in monitor_names:

    pm25 = (
        march_week_2026['uo'][
            march_week_2026['uo']['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'PM2.5'})
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    nox = (
        march_week_2026_NO['uo'][
            march_week_2026_NO['uo']['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'NOx'})
    )

    nox['NOx'] = pd.to_numeric(
        nox['NOx'],
        errors='coerce'
    )

    merged_nox = pd.merge_asof(
        pm25,
        nox,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_nox = merged_nox.dropna(subset=['PM2.5', 'NOx'])

    if not merged_nox.dropna().empty:
        valid_names.append((name, merged_nox))

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 9: Comparisons of UO-Mon sensor PM2.5 and NOx concentrations over short time interval. 

In [ ]:
n_plots = len(valid_names)
ncols = 2
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12, 8),
    constrained_layout=True
)

axes = axes.flatten()
scatter_ref = None

for i, (name, merged_nox) in enumerate(valid_names):
    
    ax = axes[i]

    # Pearson correlation
    pearson_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'UO-Mon': name,
    'Pearsons corr NOx': pearson_corr_nox,
    'Spearmans corr NOx': spearman_corr_nox
    })

    scatter_ref = ax.scatter(
        merged_nox['PM2.5'],
        merged_nox['NOx'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged_nox[['PM2.5', 'NOx']].min().min(),
    merged_nox[['PM2.5', 'NOx']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NOx')
    ax.set_title(f'{name}')
    ax.grid(True)

# hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.show()

Table 5: Pearson's and Spearman's correlations for UO-Mon sensor PM2.5 concentrations and NOx concentrations over short time interval.

In [ ]:
summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'UO-Mon',
    'Pearsons corr NOx',
    'Spearmans corr NOx'
]]

summary_df = summary_df.sort_values(
    by='Pearsons corr NOx',
    ascending=False
)

summary_df[['Pearsons corr NOx', 'Spearmans corr NOx']] = (
    summary_df[['Pearsons corr NOx', 'Spearmans corr NOx']].round(3)
)

summary_df.set_index('UO-Mon', inplace=True)

summary_df

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

## 4. Attributes affecting PM2.5 concentraions

### 4.1 Wind direction and speed

Wind data from UO-Mon sensors was taken for the month of March 2026, which turned out to be faulty as seen from Figure 10 below showing constant speed and direction over the course of the entire day/week/month. Further to this we analysed wind data from two older UO sensors in Birtley and West Denton to see if we could dicern anything interesting.

In [ ]:
uo_2026_wind = pd.read_csv("2026uptoMay27-wind-UO.csv")

uo_2026_wind["Timestamp"] = pd.to_datetime(uo_2026_wind["Timestamp"], errors="coerce")

uo_march_2026_wind = uo_2026_wind[
    (uo_2026_wind["Timestamp"] >= "2026-03-01") &
    (uo_2026_wind["Timestamp"] < "2026-03-31")
]

uo_march_2026_wind["Sensor_Name"] = (
        uo_march_2026_wind["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo_march_2026_wind["Sensor_Name"])
    )

In [ ]:
summary_results = []
valid_names = []

for name in monitor_names:

    pm25 = (
        march_2026['uo'][
            march_2026['uo']['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'PM2.5'})
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    # Wind speed
    WS = (
    uo_march_2026_wind.loc[
        (uo_march_2026_wind['Sensor_Name'] == name) &
        (uo_march_2026_wind['Variable'] == 'Wind Speed'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WS'})
    )

    WS['WS'] = pd.to_numeric(
        WS['WS'],
        errors='coerce'
    )
    
    WD = (
    uo_march_2026_wind.loc[
        (uo_march_2026_wind['Sensor_Name'] == name) &
        (uo_march_2026_wind['Variable'] == 'Wind Direction'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WD'})
)

    WD['WD'] = pd.to_numeric(
        WD['WD'],
        errors='coerce'
    )

    merged_WS = pd.merge_asof(
        pm25,
        WS,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WS = merged_WS.dropna(subset=['PM2.5', 'WS'])

    merged_WD = pd.merge_asof(
        pm25,
        WD,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WD = merged_WD.dropna(subset=['PM2.5', 'WD'])

    if not merged_WS.dropna().empty and not merged_WD.dropna().empty:
        valid_names.append((name, merged_WS, merged_WD))

Figure 10: Wind speed and direction comparisons to PM2.5 concentrations from the UO-Mon sensors.

In [ ]:
n_pairs = len(valid_names)

ncols_pairs = 2
nrows_pairs = math.ceil(n_pairs / ncols_pairs)

fig, axes = plt.subplots(
    nrows=nrows_pairs,
    ncols=ncols_pairs * 2,   # 2 plots per pair
    figsize=(16, 3 * nrows_pairs),
    constrained_layout=True
)

axes = np.atleast_2d(axes)

for i in range(n_pairs, nrows_pairs * ncols_pairs):
    pair_row = i // ncols_pairs
    pair_col = i % ncols_pairs

    ax1 = axes[pair_row, pair_col * 2]
    ax2 = axes[pair_row, pair_col * 2 + 1]

    ax1.set_visible(False)
    ax2.set_visible(False)

for i, (name, merged_WS, merged_WD) in enumerate(valid_names):

    pair_row = i // ncols_pairs
    pair_col = i % ncols_pairs

    ax1 = axes[pair_row, pair_col * 2]
    ax2 = axes[pair_row, pair_col * 2 + 1]

    ax1.scatter(
        merged_WS['PM2.5'],
        merged_WS['WS'],
        alpha=0.5,
        s = 20
    )

    ax1.set_xlabel('PM2.5')
    ax1.set_ylabel('Wind Speed')
    ax1.set_title(f'{name}', pad=10, x=0.99)
    ax1.grid(True)

    ax2.scatter(
        merged_WD['PM2.5'],
        merged_WD['WD'],
        alpha=0.5,
        s = 20
    )

    ax2.set_xlabel('PM2.5')
    ax2.set_ylabel('Wind Direction')
    ax2.set_ylim(0,360)
    ax2.grid(True)

plt.show()

The two UO sensors in Birtley and West Denton seem to have data througout 2025, simiarly to previous we looked at March 2025 data, we analysed how PM2.5 concentrations behaved in the closest UO-Mon sensors and Local sensors which seems to show a relation in wind direction. Figure 11 shows maps of the closest UO-Mon/Local sensors to the Birtley and West Denton sensors. Figure 12 shows scatter plots of PM2.5 concentration to wind speed coloured by wind direction, rose plots of PM2.5 concentration to wind direction, and binned plots of wind speed and direction coloured by PM2.5 concentration for the closest UO-Mon sensors and Figure 13 shows the same for closes Local sensors. It is especially apparent in the rose plots that the higher PM2.5 concentrations are present when the wind direction in the first quarter (0-90 degrees) with some cross over into neighbouring quarters. This is less apparent in the Local sensor comparisons, although the trend is still present, which may be a result of PM2.5 data measured every hour in comparison to the UO-Mon sensors which measure every 15 minutes. From the scatter plot and binned plot we can also see the relation to wind speed where we tend to see consistently lower PM2.5 concentrations at higher wind speeds.

In [ ]:
wind_march_2025 = timeframe_PM25_wind_2025("2025-03-30","2025-04-05")

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 11: Maps of UO-Mesh sensors and Local sensors to UO wind sensors in Birtley and West Denton.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

all_sensors = pd.concat(
    [
        wind_march_2025['monitor_locations'],
        wind_march_2025['wind_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    wind_march_2025['closest_wind_monitor'],
    wind_march_2025['wind_locations'],
    wind_march_2025['monitor_locations'],
    all_locations=all_sensors,
    label_1='UO Wind',
    label_2='UO-Mon',
    title='Closest UO wind to UO-Mon Sensors',
    ax=axes[0]
)

all_sensors = pd.concat(
    [
        wind_march_2025['local_locations'],
        wind_march_2025['wind_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    wind_march_2025['closest_wind_local'],
    wind_march_2025['wind_locations'],
    wind_march_2025['local_locations'],
    all_locations=all_sensors,
    label_1='UO wind',
    label_2='Local sensors',
    title='Closest UO wind to Local sensors',
    ax=axes[1]
)

plt.tight_layout()
plt.show()

In [ ]:
summary_results = []
valid_rows = []

for _, row in wind_march_2025['closest_wind_monitor'].iterrows():

    wind_name = row['Sensor 1']
    monitor_name = row['Sensor 2']

    pm25 = (
        wind_march_2025['uo'][
            wind_march_2025['uo']['Sensor_Name'] == monitor_name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'PM2.5'})
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    WS = (
    wind_march_2025['wind'].loc[
        (wind_march_2025['wind']['Sensor_Name'] == wind_name) &
        (wind_march_2025['wind']['Variable'] == 'Wind Speed'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WS'})
    )

    WS['WS'] = pd.to_numeric(
        WS['WS'],
        errors='coerce'
    )
    
    WD = (
    wind_march_2025['wind'].loc[
        (wind_march_2025['wind']['Sensor_Name'] == wind_name) &
        (wind_march_2025['wind']['Variable'] == 'Wind Direction'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WD'})
    )

    WD['WD'] = pd.to_numeric(
        WD['WD'],
        errors='coerce'
    )

    merged_WS = pd.merge_asof(
        pm25,
        WS,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WS = merged_WS.dropna(subset=['PM2.5', 'WS'])

    merged_WD = pd.merge_asof(
        pm25,
        WD,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WD = merged_WD.dropna(subset=['PM2.5', 'WD'])

    if not merged_WS.dropna().empty and not merged_WD.dropna().empty:
        valid_rows.append((row, wind_name, monitor_name, merged_WS, merged_WD))

Figure 12: Scatter plots of PM2.5 concentration to wind speed coloured by wind direction, rose plots of PM2.5 concentration to wind direction, and binned plots of wind speed and direction coloured by PM2.5 concentration for the closest UO-Mon sensors to UO wind sensors in Birtley and West Denton

In [ ]:
n_pairs = len(valid_rows)

fig, axes = plt.subplots(
    nrows=n_pairs,
    ncols=3,
    figsize=(18, 4 * n_pairs),
    constrained_layout=True
)

if n_pairs == 1:
    axes = np.array([axes])

for i, (row, wind_name, monitor_name, merged_WS, merged_WD) in enumerate(valid_rows):

    ax1, ax2, ax3 = axes[i]

    # Pearson correlation
    pearson_corr_WS = merged_WS[['PM2.5', 'WS']].corr(method='pearson').iloc[0, 1]
    pearson_corr_WD = merged_WD[['PM2.5', 'WD']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_WS = merged_WS[['PM2.5', 'WS']].corr(method='spearman').iloc[0, 1]
    spearman_corr_WD = merged_WD[['PM2.5', 'WD']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'Wind sensor': wind_name,    
    'UO monitor sensor': monitor_name,
    'Pearsons corr WS': pearson_corr_WS,
    'Spearmans corr WS': spearman_corr_WS,
    'Pearsons corr WD': pearson_corr_WD,
    'Spearmans corr WD': spearman_corr_WD
    })

    sc = ax1.scatter(
        merged_WS['PM2.5'],
        merged_WS['WS'],
        alpha=0.5,
        c=merged_WD["WD"],
        cmap="twilight",
        s=20
    )

    ax1.set_xlabel('PM2.5')
    ax1.set_ylabel('Wind Speed')
    ax1.set_title(f'{monitor_name} vs {wind_name}\nPM2.5–WS colored by WD')
    ax1.grid(True)
    plt.colorbar(sc, ax=ax1, label="Wind Direction (°)")

    merged_WD = merged_WD.copy()
    merged_WD["dir_bin"] = (merged_WD["WD"] // 10) * 10

    dir_summary = merged_WD.groupby("dir_bin")["PM2.5"].mean().reset_index()

    theta = np.deg2rad(dir_summary["dir_bin"])

    ax2 = plt.subplot(n_pairs, 3, 3*i + 2, projection='polar')

    ax2.bar(
        theta,
        dir_summary["PM2.5"],
        width=np.deg2rad(10),
        color="steelblue",
        alpha=0.7
    )

    ax2.set_title("Mean PM2.5 by wind direction")

    merged_WD["ws_bin"] = (merged_WS["WS"] // 1) * 1

    pivot = merged_WD.pivot_table(
        values="PM2.5",
        index="ws_bin",
        columns="dir_bin",
        aggfunc="mean"
    )

    im = ax3.imshow(
        pivot.values,
        aspect='auto',
        origin='lower',
        cmap="magma"
    )

    ax3.set_title("PM2.5 by wind direction & speed")
    ax3.set_xlabel("Wind Direction (°)")
    ax3.set_ylabel("Wind Speed")

    ax3.set_xticks(np.arange(len(pivot.columns)))
    ax3.set_xticklabels(pivot.columns, rotation=90)

    ax3.set_yticks(np.arange(len(pivot.index)))
    ax3.set_yticklabels(pivot.index)

    plt.colorbar(im, ax=ax3, label="PM2.5")

plt.show()

In [ ]:
summary_results = []
valid_rows = []

for _, row in wind_march_2025['closest_wind_local'].iterrows():

    wind_name = row['Sensor 1']
    local_name = row['Sensor 2']

    pm25 = (
        wind_march_2025['local'][
            wind_march_2025['local']['Sensor_Name'] == local_name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    WS = (
    wind_march_2025['wind'].loc[
        (wind_march_2025['wind']['Sensor_Name'] == wind_name) &
        (wind_march_2025['wind']['Variable'] == 'Wind Speed'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WS'})
    )

    WS['WS'] = pd.to_numeric(
        WS['WS'],
        errors='coerce'
    )
    
    WD = (
    wind_march_2025['wind'].loc[
        (wind_march_2025['wind']['Sensor_Name'] == wind_name) &
        (wind_march_2025['wind']['Variable'] == 'Wind Direction'),
        ['Timestamp', 'Value']
    ]
    .sort_values('Timestamp')
    .rename(columns={'Value': 'WD'})
    )

    WD['WD'] = pd.to_numeric(
        WD['WD'],
        errors='coerce'
    )

    merged_WS = pd.merge_asof(
        pm25,
        WS,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WS = merged_WS.dropna(subset=['PM2.5', 'WS'])

    merged_WD = pd.merge_asof(
        pm25,
        WD,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_WD = merged_WD.dropna(subset=['PM2.5', 'WD'])

    if not merged_WS.dropna().empty and not merged_WD.dropna().empty:
        valid_rows.append((row, wind_name, local_name, merged_WS, merged_WD))

In [ ]:
display(HTML("<div style='page-break-after: always;'></div>"))

Figure 13: Scatter plots of PM2.5 concentration to wind speed coloured by wind direction, rose plots of PM2.5 concentration to wind direction, and binned plots of wind speed and direction coloured by PM2.5 concentration for the closest Local sensors to UO wind sensors in Birtley and West Denton

In [ ]:
n_pairs = len(valid_rows)

fig, axes = plt.subplots(
    nrows=n_pairs,
    ncols=3,
    figsize=(18, 4 * n_pairs),
    constrained_layout=True
)

if n_pairs == 1:
    axes = np.array([axes])

for i, (row, wind_name, local_name, merged_WS, merged_WD) in enumerate(valid_rows):

    ax1, ax2, ax3 = axes[i]

    # Pearson correlation
    pearson_corr_WS = merged_WS[['PM2.5', 'WS']].corr(method='pearson').iloc[0, 1]
    pearson_corr_WD = merged_WD[['PM2.5', 'WD']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_WS = merged_WS[['PM2.5', 'WS']].corr(method='spearman').iloc[0, 1]
    spearman_corr_WD = merged_WD[['PM2.5', 'WD']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'Wind sensor': wind_name,    
    'Local sensor': local_name,
    'Pearsons corr WS': pearson_corr_WS,
    'Spearmans corr WS': spearman_corr_WS,
    'Pearsons corr WD': pearson_corr_WD,
    'Spearmans corr WD': spearman_corr_WD
    })

    sc = ax1.scatter(
        merged_WS['PM2.5'],
        merged_WS['WS'],
        alpha=0.5,
        c=merged_WD["WD"],
        cmap="twilight",
        s=20
    )

    ax1.set_xlabel('PM2.5')
    ax1.set_ylabel('Wind Speed')
    ax1.set_title(f'{local_name} vs {wind_name}\nPM2.5–WS colored by WD')
    ax1.grid(True)
    plt.colorbar(sc, ax=ax1, label="Wind Direction (°)")

    merged_WD = merged_WD.copy()
    merged_WD["dir_bin"] = (merged_WD["WD"] // 10) * 10

    dir_summary = merged_WD.groupby("dir_bin")["PM2.5"].mean().reset_index()

    theta = np.deg2rad(dir_summary["dir_bin"])

    ax2 = plt.subplot(n_pairs, 3, 3*i + 2, projection='polar')

    ax2.bar(
        theta,
        dir_summary["PM2.5"],
        width=np.deg2rad(10),
        color="steelblue",
        alpha=0.7
    )

    ax2.set_title("Mean PM2.5 by wind direction")

    merged_WD["ws_bin"] = (merged_WS["WS"] // 1) * 1

    pivot = merged_WD.pivot_table(
        values="PM2.5",
        index="ws_bin",
        columns="dir_bin",
        aggfunc="mean"
    )

    im = ax3.imshow(
        pivot.values,
        aspect='auto',
        origin='lower',
        cmap="magma"
    )

    ax3.set_title("PM2.5 by wind direction & speed")
    ax3.set_xlabel("Wind Direction (°)")
    ax3.set_ylabel("Wind Speed")

    ax3.set_xticks(np.arange(len(pivot.columns)))
    ax3.set_xticklabels(pivot.columns, rotation=90)

    ax3.set_yticks(np.arange(len(pivot.index)))
    ax3.set_yticklabels(pivot.index)

    plt.colorbar(im, ax=ax3, label="PM2.5")

plt.show()

### 4.2 Altitude

We have used Newcastle upon Tyne extent of the DEFRA/Local/UO-Mon sensor area Ordinate Survey digital elevation map to evaluate whether any concentration level could be attributed to altitude of the sensors. From Figuare 14 where we look and mean and standard deviations for each sensor PM2.5 data over the course of the month with their altitude we see that there is a small positive correlation for mean and a moderate positive correlation for the standard deviation which indicates the higher altitude could affect higher variability in PM2.5 concentrations. 

In [ ]:
dem = rasterio.open("Newcastle_altitude/combined_altitude_raster_wgs84.tif")

In [ ]:
march_2026['monitor_locations']['Altitude'] = march_2026['monitor_locations'].apply(
    lambda row: list(
        dem.sample([
            (
                row['Long'],
                row['Lat']
            )
        ])
    )[0][0],
    axis=1
)

march_2026['defra_locations']['Altitude'] = march_2026['defra_locations'].apply(
    lambda row: list(
        dem.sample([
            (
                row['Long'],
                row['Lat']
            )
        ])
    )[0][0],
    axis=1
)

march_2026['local_locations']['Altitude'] = march_2026['local_locations'].apply(
    lambda row: list(
        dem.sample([
            (
                row['Long'],
                row['Lat']
            )
        ])
    )[0][0],
    axis=1
)

In [ ]:
march_2026['defra']['PM2.5'] = pd.to_numeric(
    march_2026['defra']['PM2.5'],
    errors='coerce'
)

march_2026['local']['PM2.5'] = pd.to_numeric(
    march_2026['local']['PM2.5'],
    errors='coerce'
)

mean_pm25_uo = (march_2026['uo']
                .groupby('Sensor_Name')['Value']
                .agg(
                    mean='mean',
                    std='std',
                    median='median',
                    minimum='min',
                    maximum='max',
                    count='count'
                )
                .reset_index()
)

mean_pm25_uo = mean_pm25_uo.merge(
    march_2026['monitor_locations'][['Sensor_Name', 'Altitude']],
    on='Sensor_Name'
)

mean_pm25_defra = (march_2026['defra']
                .groupby('Sensor_Name')['PM2.5']
                .agg(
                    mean='mean',
                    std='std',
                    median='median',
                    minimum='min',
                    maximum='max',
                    count='count'
                )
                .reset_index()
                )

mean_pm25_defra = mean_pm25_defra.merge(
    march_2026['defra_locations'][['Sensor_Name', 'Altitude']],
    on='Sensor_Name'
)

mean_pm25_local = (march_2026['local']
                .groupby('Sensor_Name')['PM2.5']
                .agg(
                    mean='mean',
                    std='std',
                    median='median',
                    minimum='min',
                    maximum='max',
                    count='count'
                )
                .reset_index()
                )

mean_pm25_local = mean_pm25_local.merge(
    march_2026['local_locations'][['Sensor_Name', 'Altitude']],
    on='Sensor_Name'
)

In [ ]:
combined_mean = pd.concat([
    mean_pm25_uo[['mean', 'Altitude']],
    mean_pm25_defra[['mean', 'Altitude']],
    mean_pm25_local[['mean', 'Altitude']]
], ignore_index=True)

combined_std = pd.concat([
    mean_pm25_uo[['std', 'Altitude']],
    mean_pm25_defra[['std', 'Altitude']],
    mean_pm25_local[['std', 'Altitude']]
], ignore_index=True)

 # Pearson correlation
pearson_corr_mean = combined_mean[['mean', 'Altitude']].corr(method='pearson').iloc[0, 1]
pearson_corr_std = combined_std[['std', 'Altitude']].corr(method='pearson').iloc[0, 1]

# Spearman correlation
spearman_corr_mean = combined_mean[['mean', 'Altitude']].corr(method='spearman').iloc[0, 1]
spearman_corr_std = combined_std[['std', 'Altitude']].corr(method='spearman').iloc[0, 1]


Figure 14: Altitude of DEFRA/Local/UO-Mon sensors with mean and standard deviation in PM2.5 concentration over long time interval 

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Mean
axs[0].scatter(mean_pm25_uo['mean'],
               mean_pm25_uo['Altitude'],
               label='UO',
               alpha=0.7)

axs[0].scatter(mean_pm25_defra['mean'],
               mean_pm25_defra['Altitude'],
               label='DEFRA',
               alpha=0.7)

axs[0].scatter(mean_pm25_local['mean'],
               mean_pm25_local['Altitude'],
               label='Local',
               alpha=0.7)

axs[0].text(
    0.02, 0.98,
    f'Pearson = {pearson_corr_mean:.2f} \n'
    f'Spearman = {spearman_corr_mean:.2f}',
    transform=axs[0].transAxes,
    va='top',
    bbox=dict(facecolor='white', alpha=0.8)
)

axs[0].set_xlabel('Mean PM2.5')
axs[0].set_ylabel('Altitude (m)')
axs[0].set_title('Mean PM2.5 with Altitude')
axs[0].legend()
axs[0].grid(True)

# Standard deviation
axs[1].scatter(mean_pm25_uo['std'],
               mean_pm25_uo['Altitude'],
               label='UO',
               alpha=0.7)

axs[1].scatter(mean_pm25_defra['std'],
               mean_pm25_defra['Altitude'],
               label='DEFRA',
               alpha=0.7)

axs[1].scatter(mean_pm25_local['std'],
               mean_pm25_local['Altitude'],
               label='Local',
               alpha=0.7)

axs[1].text(
    0.02, 0.98,
    f'Pearson = {pearson_corr_std:.2f} \n'
    f'Spearman = {spearman_corr_std:.2f}',
    transform=axs[1].transAxes,
    va='top',
    bbox=dict(facecolor='white', alpha=0.8)
)

axs[1].set_xlabel('Standard Deviation PM2.5')
axs[1].set_title('PM2.5 Standard deviation with Altitude')
axs[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
alt_pm25_uo = march_2026['uo'].merge(
    march_2026['monitor_locations'][['Sensor_Name', 'Altitude']],
    on='Sensor_Name'
)

alt_pm25_defra = march_2026['defra'].merge(
    march_2026['defra_locations'][['Sensor_Name', 'Altitude']],
    on='Sensor_Name'
)

alt_pm25_local = march_2026['local'].merge(
    march_2026['local_locations'][['Sensor_Name', 'Altitude']],
    on='Sensor_Name'
)

alt_pm25_local['PM2.5'] = pd.to_numeric(
    alt_pm25_local['PM2.5'],
    errors='coerce'
)

alt_pm25_defra['PM2.5'] = pd.to_numeric(
    alt_pm25_defra['PM2.5'],
    errors='coerce'
)

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as colors

fig, ax = plt.subplots(figsize=(12,8))

all_altitudes = pd.concat([
    alt_pm25_uo['Altitude'],
    alt_pm25_local['Altitude'],
    alt_pm25_defra['Altitude']
])

norm = colors.Normalize(vmin=all_altitudes.min(),
                        vmax=all_altitudes.max())
cmap = cm.viridis

for name in monitor_names:
    df = alt_pm25_uo[alt_pm25_uo['Sensor_Name'] == name]
    altitude = df['Altitude'].iloc[0]
    ax.plot(df['Timestamp'],
            df['Value'],
            color=cmap(norm(altitude)),
            label=name)

for name in local_names:
    df = alt_pm25_local[alt_pm25_local['Sensor_Name'] == name]
    altitude = df['Altitude'].iloc[0]
    ax.plot(df['Timestamp'],
            df['PM2.5'],
            color=cmap(norm(altitude)),
            label=name)

# DEFRA sensors
for name in defra_names:
    df = alt_pm25_defra[alt_pm25_defra['Sensor_Name'] == name]
    altitude = df['Altitude'].iloc[0]
    ax.plot(df['Timestamp'],
            df['PM2.5'],
            color=cmap(norm(altitude)),
            label=name)

# Add colourbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Altitude (m)')

ax.set_xlabel('Timestamp')
ax.set_ylabel('PM2.5')
ax.set_title('PM2.5 concentrations coloured by altitude')
ax.grid(True)

plt.show()

In [ ]:
print(alt_pm25_uo.dtypes)
print(alt_pm25_local.dtypes)
print(alt_pm25_defra.dtypes)

### 4.2 Temperature

We compared UO-Mon sensor PM2.5 with the same sensor temperature data over March 2026 in order to discern if there is relationship as our analysis with the NOx and NO2 concentrations could suggest that other factors may be of interest. In Figure 15, we can see scatter plots for temperature comparison with PM2.5 concentration for the UO-Mon sensors which, alongside low correlation values seen in Table 6, suggest very little relation between temperature and PM2.5 concentrations for at least a monthly time scale.

In [ ]:
uo_mon_2026_temp = pd.read_csv("2026March-Temperature-UO_Mon.csv")

uo_mon_2026_temp["Sensor_Name"] = (
        uo_mon_2026_temp["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo_mon_2026_temp["Sensor_Name"])
    )

monitor_names = set(march_2026['monitor_locations']['Sensor_Name'].unique()) 

In [ ]:
summary_results = []
valid_names = []

for name in monitor_names:

    pm25 = (
        march_2026['uo'][
            march_2026['uo']['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'PM2.5'})
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    temp = (
        uo_mon_2026_temp[
            uo_mon_2026_temp['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'Temperature'})
    )

    temp['Temperature'] = pd.to_numeric(
        temp['Temperature'],
        errors='coerce'
    )

    pm25['Timestamp'] = pd.to_datetime(pm25['Timestamp'], errors='coerce')
    temp['Timestamp'] = pd.to_datetime(temp['Timestamp'], errors='coerce')

    merged_temp = pd.merge_asof(
        pm25,
        temp,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_temp = merged_temp.dropna(subset=['PM2.5', 'Temperature'])

    if not merged_temp.dropna().empty:
        valid_names.append((name, merged_temp))

Figure 15: Comaprsion on PM2.5 concentration and temperature from UO-Mon sensors.

In [ ]:
n_plots = len(valid_names)
ncols = 2
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12, 8),
    constrained_layout=True
)

axes = axes.flatten()
scatter_ref = None

for i, (name, merged_temp) in enumerate(valid_names):
    
    ax = axes[i]

    # Pearson correlation
    pearson_corr_temp = merged_temp[['PM2.5', 'Temperature']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_temp = merged_temp[['PM2.5', 'Temperature']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'UO-Mon': name,
    'Pearsons corr Temp': pearson_corr_temp,
    'Spearmans corr Temp': spearman_corr_temp
    })

    scatter_ref = ax.scatter(
        merged_temp['PM2.5'],
        merged_temp['Temperature'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged_temp[['PM2.5', 'Temperature']].min().min(),
    merged_temp[['PM2.5', 'Temperature']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('Temperature')
    ax.set_title(f'{name}')
    ax.grid(True)

# hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.show()

Table 6: Pearson's and Spearman's correlations for UO-Mon sensor PM2.5 concentrations and temperature over long time interval.

In [ ]:
summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'UO-Mon',
    'Pearsons corr Temp',
    'Spearmans corr Temp'
]]

summary_df = summary_df.sort_values(
    by='Pearsons corr Temp',
    ascending=False
)

summary_df[['Pearsons corr Temp', 'Spearmans corr Temp']] = (
    summary_df[['Pearsons corr Temp', 'Spearmans corr Temp']].round(3)
)

summary_df.set_index('UO-Mon', inplace=True)

summary_df